# Fusion : One Explainable Priority Product

This is the final step. It joins the three modules plus the live TomTom signal into a single
ranked queue where **every number carries a one-line reason** - the "why is this flagged"
panel the brief asks for.

- **Module 1** - uncertainty-aware priority band (how urgent, and how sure we are).
- **Module 2** - calibrated probability of a severe incident next week (forward-looking).
- **Module 3** - divergence / blind-spot flag (kept as a *separate* investigate list, not
  folded into the score, so a data gap never silently inflates urgency).
- **TomTom** - live congestion and active incidents right now (present-day context).

Composite score = a weighted blend of standardized historical priority, forward risk, and a
live-congestion nudge. Module 3 rides alongside as a flag.

### 1. Load every module output

In [ ]:
import pandas as pd
import numpy as np
import json, os

def load(path):
    return pd.read_json(path) if os.path.exists(path) else None

m1 = load("module1_priority.json")        # priority band
m2 = load("module2_risk.json")            # calibrated weekly risk
m3 = load("module3_divergence.json")      # blind-spot divergence
tf = load("traffic_features.json")        # live TomTom (optional, top-N only)

# module 3 ships in two shapes; normalize to location_id + divergence_category
if m3 is not None:
    if "location" in m3.columns and "location_id" not in m3.columns:
        m3 = m3.rename(columns={"location": "location_id"})
    if "status" in m3.columns and "divergence_category" not in m3.columns:
        m3["divergence_category"] = m3["status"].map(
            {"blindspot": "potential_blind_spot", "underreported": "potential_reporting_gap"}
        ).fillna("signals_aligned")

for df in [m1, m2, m3, tf]:
    if df is not None:
        df["location_id"] = df["location_id"].astype(str).str.strip()

print("module1:", None if m1 is None else m1.shape,
      "| module2:", None if m2 is None else m2.shape,
      "| module3:", None if m3 is None else m3.shape,
      "| traffic:", None if tf is None else tf.shape)


### 2. Join on `location_id`

Module 1 is the spine (one row per location). Module 2 and 3 join in; live traffic is only
present for the top-N enriched locations, so those columns are null elsewhere - that is
expected and handled downstream.

In [ ]:
base = m1[["location_id", "zone", "latitude", "longitude", "road_type", "road_condition",
           "pedestrian_activity", "near_school", "near_hospital",
           "predicted", "band_low", "band_high", "confidence", "explanation"]].rename(
    columns={"predicted": "m1_priority", "confidence": "m1_confidence",
             "explanation": "m1_why"})

if m2 is not None:
    base = base.merge(m2[["location_id", "risk_prob", "risk_label", "explanation"]].rename(
        columns={"explanation": "m2_why"}), on="location_id", how="left")
if m3 is not None:
    dcol = "divergence" if "divergence" in m3.columns else None
    keep = ["location_id", "divergence", "divergence_category", "explanation"]
    keep = [c for c in keep if c in m3.columns]
    base = base.merge(m3[keep].rename(columns={"explanation": "m3_why",
           "divergence_category": "m3_flag"}), on="location_id", how="left")
if tf is not None:
    base = base.merge(tf[["location_id", "congestion_ratio", "roadClosure",
           "live_incident_count", "max_live_delay"]], on="location_id", how="left")

print("fused:", base.shape)
base.head(3)


### 3. Composite score

We z-score the historical priority (Module 1) and the forward risk (Module 2) so neither
dominates by scale, then add a small live-congestion nudge (`1 - congestion_ratio`, so a
jammed road ranks a little higher). Module 3 is **not** added - it stays a separate flag.

In [ ]:
def z(s):
    s = pd.to_numeric(s, errors="coerce")
    sd = s.std(ddof=0)
    return (s - s.mean()) / sd if sd and not np.isnan(sd) else pd.Series(0.0, index=s.index)

base["z_priority"] = z(base["m1_priority"])
base["z_risk"] = z(base["risk_prob"]) if "risk_prob" in base else 0.0

# live jam factor: 0 when free-flowing / unknown, up to 1 when fully jammed
cr = base["congestion_ratio"] if "congestion_ratio" in base else pd.Series(np.nan, index=base.index)
base["jam_factor"] = (1 - pd.to_numeric(cr, errors="coerce")).clip(0, 1).fillna(0)

W_PRIORITY, W_RISK, W_JAM = 0.5, 0.4, 0.1
base["composite"] = (W_PRIORITY * base["z_priority"]
                     + W_RISK * base["z_risk"]
                     + W_JAM * base["jam_factor"])
base["composite_rank"] = base["composite"].rank(ascending=False, method="first").astype(int)
base = base.sort_values("composite_rank").reset_index(drop=True)
base[["location_id", "z_priority", "z_risk", "jam_factor", "composite", "composite_rank"]].head(8)


### 4. The explainable card - one plain-English line per signal

This is the product. Each location shows its band, its calibrated forward risk, any blind-spot
flag, and live context, each with a reason - never a bare black-box number.

In [ ]:
def card(r):
    lines = []
    lines.append(f"Priority band: {r['band_low']:.2f} to {r['band_high']:.2f} "
                 f"({r['m1_confidence']} confidence)")
    if "risk_prob" in base and pd.notna(r.get("risk_prob")):
        lines.append(f"Risk next week: {round(r['risk_prob']*100)}% chance of a severe "
                     f"incident ({r.get('risk_label','')})")
    if "m3_flag" in base and pd.notna(r.get("m3_flag")) and r["m3_flag"] != "signals_aligned":
        dv = r.get("divergence")
        tail = f" (divergence {dv:+.2f})" if pd.notna(dv) else ""
        lines.append(f"Data check: {r['m3_flag'].replace('_',' ')}{tail}")
    if pd.notna(r.get("congestion_ratio")):
        pct = round(r["congestion_ratio"] * 100)
        closed = ", road currently closed" if r.get("roadClosure") else ""
        li = int(r["live_incident_count"]) if pd.notna(r.get("live_incident_count")) else 0
        lines.append(f"Live: moving at {pct}% of free-flow speed, "
                     f"{li} active incident(s) nearby{closed}")
    return lines

base["why_card"] = base.apply(lambda r: card(r), axis=1)

# show the top 3 cards
for _, r in base.head(3).iterrows():
    print(f"#{r['composite_rank']}  {r['location_id']}  ({r['zone']}, {r['road_type']})")
    for line in r["why_card"]:
        print("   -", line)
    print()


### 5. Separate blind-spot list (Module 3)

Per the brief, locations where citizens and the official record disagree are surfaced as a
distinct "investigate the data" list, not mixed into the fix-first queue.

In [ ]:
if "m3_flag" in base:
    gaps = base[base["m3_flag"].isin(["potential_blind_spot", "potential_reporting_gap"])].copy()
    if "divergence" in gaps and gaps["divergence"].notna().any():
        gaps = gaps.reindex(gaps["divergence"].abs().sort_values(ascending=False).index)
        cols = ["location_id", "zone", "m3_flag", "divergence"]
    else:
        cols = ["location_id", "zone", "m3_flag"]
    print(f"{len(gaps)} locations flagged for data review")
    print(gaps[cols].head(10).to_string(index=False))
else:
    print("Module 3 output not present; skipping blind-spot list.")


### 6. Export the composite priority queue

In [ ]:
out_cols = ["location_id", "zone", "latitude", "longitude", "road_type", "road_condition",
            "pedestrian_activity", "m1_priority", "band_low", "band_high", "m1_confidence",
            "risk_prob", "risk_label", "divergence", "m3_flag",
            "congestion_ratio", "roadClosure", "live_incident_count", "max_live_delay",
            "composite", "composite_rank", "why_card"]
out_cols = [c for c in out_cols if c in base.columns]
out = base[out_cols].replace([np.inf, -np.inf], np.nan)
out.to_json("composite_priority.json", orient="records", indent=2)
print(f"Exported {len(out)} locations -> composite_priority.json")
print(out.head(2).to_json(orient="records", indent=2))
